# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

---

In [1]:
# importar librerías
import pandas as pd

In [2]:
# cargar archivos
orders = pd.read_csv('/datasets/rappiplus_orders_raw.csv')# tu código aquí
catalog = pd.read_csv('/datasets/rappiplus_catalog.csv')# tu código aquí
marketing = pd.read_csv('/datasets/rappiplus_marketing_spend.csv')# tu código aquí

In [3]:
orders.head(5)# explorar datasets
# tu código aquí

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [4]:
catalog.head(5)

,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


In [5]:
marketing.head(5)

,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

---

In [6]:
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'])
orders.info()
catalog.info()
marketing['fecha'] = pd.to_datetime(marketing['fecha'])
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           25100 non-null  object        
 1   id_usuario          25100 non-null  object        
 2   fecha_hora_pedido   25100 non-null  datetime64[ns]
 3   pais                24800 non-null  object        
 4   dispositivo         25080 non-null  object        
 5   fuente_referencia   25070 non-null  object        
 6   nombre_producto     25070 non-null  object        
 7   categoria_producto  25020 non-null  object        
 8   cantidad            25050 non-null  float64       
 9   precio_unitario     25050 non-null  float64       
 10  monto_descuento     25050 non-null  float64       
 11  monto_total         25100 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.3+ MB
<class 'pandas.core.frame.DataFrame'

In [7]:
print(orders.describe())
cols_num = ['cantidad'] 
for c in cols_num:
    print(c, '-> negativos:', (orders[c] < 0).sum(), '| ceros:', (orders[c] == 0).sum())
orders[orders['cantidad'] <= 0]
orders['cantidad'] = orders['cantidad'].abs()


cols_num = ['monto_total']  
for c in cols_num:
    print(c, '-> negativos:', (orders[c] < 0).sum(), '| ceros:', (orders[c] == 0).sum())
orders[orders['monto_total'] <= 0]
orders['monto_total'] = orders['monto_total'].abs()
#se observan numeros negativos en la columna cantidad, se conservan filas con valores absolutos.

           cantidad  precio_unitario  monto_descuento   monto_total
count  25050.000000     25050.000000     25050.000000  2.510000e+04
mean       7.092735       259.305549         4.500798  2.072680e+03
std      296.277003       138.726461         5.223010  9.894995e+04
min       -2.000000        20.030000         0.000000 -4.926500e+02
25%        1.000000       138.377500         0.000000  1.805075e+02
50%        2.000000       258.715000         0.000000  3.417500e+02
75%        2.000000       380.332500        10.000000  5.185800e+02
max    20000.000000       499.960000        15.000000  8.840200e+06
cantidad -> negativos: 4 | ceros: 0
monto_total -> negativos: 4 | ceros: 0


In [8]:
print('Duplicados completos:', orders.duplicated().sum())
print('Duplicados por id_pedido:', orders.duplicated(subset=['id_pedido']).sum())

# Míralos antes de borrar
orders[orders.duplicated(subset=['id_pedido'], keep=False)].sort_values('id_pedido')

orders = orders.drop_duplicates(subset=['id_pedido']).reset_index(drop=True)

Duplicados completos: 100
Duplicados por id_pedido: 100


In [9]:
#revisión variables categoricas
for c in ['nombre_producto', 'categoria_producto', 'proveedor']:
    catalog[c] = catalog[c].str.strip()
    print(f'\n--- {c} ---')
    print(catalog[c].value_counts(dropna=False).head(20))


--- nombre_producto ---
Laptop-Gaming-16GB      1
Phone-Pro-128GB         1
Jacket-Winter-M         1
Vacuum-Pro-Black        1
Sneakers-Urban-42       1
Tablet-Standard-64GB    1
Blender-XL-Red          1
Name: nombre_producto, dtype: int64

--- categoria_producto ---
Electrónica    3
Hogar          2
Moda           2
Name: categoria_producto, dtype: int64

--- proveedor ---
Greene-Smith               1
Mcmillan-Rhodes            1
King Ltd                   1
Bowers LLC                 1
Fuller, Pena and Myers     1
Rivera, Carr and Finley    1
Long-Reid                  1
Name: proveedor, dtype: int64


In [10]:
print(marketing['canal'].isna().sum())      

#relleno de columna canal con la columna id_campaña
canal_desde_id = marketing['id_campaña'].str.rsplit('_', n=1).str[0]
marketing['canal'] = marketing['canal'].fillna(canal_desde_id)

print(marketing['canal'].isna().sum())  
print(marketing['canal'].value_counts())  
marketing.info()
orders.info()

101
0
paid_search    540
organic        540
social         540
Name: canal, dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   fecha       1620 non-null   datetime64[ns]
 1   pais        1620 non-null   object        
 2   id_campaña  1620 non-null   object        
 3   canal       1620 non-null   object        
 4   gasto       1620 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 63.4+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           25000 non-null  object        
 1   id_usuario          25000 non-null  object        
 2   fecha_hora_pedido   25000 non-null  datetime64[ns]
 3   pai

In [11]:
marketing.info()
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   fecha       1620 non-null   datetime64[ns]
 1   pais        1620 non-null   object        
 2   id_campaña  1620 non-null   object        
 3   canal       1620 non-null   object        
 4   gasto       1620 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 63.4+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           25000 non-null  object        
 1   id_usuario          25000 non-null  object        
 2   fecha_hora_pedido   25000 non-null  datetime64[ns]
 3   pais                24700 non-null  object        
 4   dispositivo         24980 non-null  

In [12]:
# Quedaron 50 pedidos (0.2%) con cantidad, 
# precio_unitario y monto_descuento vacíos a la vez
# no se podían reconstruir a partir de monto_total. Se eliminaron.
cols = ['cantidad', 'precio_unitario', 'monto_descuento']
antes = len(orders)
revenue_antes = orders['monto_total'].sum()

orders = orders.dropna(subset=cols).reset_index(drop=True)
orders['cantidad'] = orders['cantidad'].astype(int)

print(f'Filas: {antes} -> {len(orders)}')
print(f'Revenue: {revenue_antes:,.2f} -> {orders["monto_total"].sum():,.2f}')
print(orders.isna().sum().sum()) 

Filas: 25000 -> 24950
Revenue: 51,989,756.86 -> 51,968,128.86
410


In [13]:
chequeo = orders.merge(catalog[['nombre_producto', 'costo_unitario']], on='nombre_producto', how='left')
chequeo['bajo_costo'] = chequeo['precio_unitario'] < chequeo['costo_unitario']

resumen = chequeo.groupby('nombre_producto').agg(
    pedidos=('id_pedido', 'count'),
    precio_min=('precio_unitario', 'min'),
    precio_prom=('precio_unitario', 'mean'),
    precio_max=('precio_unitario', 'max'),
    costo=('costo_unitario', 'first'),
    pct_bajo_costo=('bajo_costo', 'mean')).round(2)

print(resumen)

                      pedidos  precio_min  precio_prom  precio_max   costo  \
nombre_producto                                                              
Blender-XL-Red           4176       20.06       259.16      499.94  176.64   
Jacket-Winter-M          4166       20.06       259.82      499.93  189.31   
Laptop-Gaming-16GB       2778       20.24       261.36      499.95  280.68   
Phone-Pro-128GB          2737       20.08       259.54      499.86   10.12   
Sneakers-Urban-42        4129       20.11       257.23      499.91   17.21   
Tablet-Standard-64GB     2764       20.03       257.53      499.89   25.21   
Vacuum-Pro-Black         4170       20.37       261.02      499.96   16.60   

                      pct_bajo_costo  
nombre_producto                       
Blender-XL-Red                  0.33  
Jacket-Winter-M                 0.36  
Laptop-Gaming-16GB              0.54  
Phone-Pro-128GB                 0.00  
Sneakers-Urban-42               0.00  
Tablet-Standard-64GB    

In [14]:
catalog.info()
print(catalog)

print('Duplicados por producto:', catalog.duplicated(subset=['nombre_producto']).sum())
print('Costos <= 0:', (catalog['costo_unitario'] <= 0).sum())
print('Nulos:', catalog.isna().sum().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes
        nombre_producto categoria_producto  costo_unitario  \
0    Laptop-Gaming-16GB        Electrónica          280.68   
1       Phone-Pro-128GB        Electrónica           10.12   
2  Tablet-Standard-64GB        Electrónica           25.21   
3        Blender-XL-Red              Hogar          176.64   
4      Vacuum-Pro-Black              Hogar           16.60   
5     Sneakers-Urban-42               Moda           17.21   
6       Jacket-Winter-M               Moda          189.31   

                 proveedor  
0   Fuller, Pena and My

In [15]:
print(marketing.isna().sum())

print('Gasto < 0:', (marketing['gasto'] < 0).sum())
print('Gasto == 0:', (marketing['gasto'] == 0).sum())
print(marketing['gasto'].describe())

print(marketing['fecha'].min(), marketing['fecha'].max())
print('Duplicados por fecha+pais+canal:', marketing.duplicated(subset=['fecha', 'pais', 'canal']).sum())
print(marketing['pais'].value_counts())
print(marketing['canal'].value_counts())

fecha         0
pais          0
id_campaña    0
canal         0
gasto         0
dtype: int64
Gasto < 0: 0
Gasto == 0: 0
count    1620.00000
mean     1772.74292
std       734.43294
min       501.11000
25%      1128.03000
50%      1782.42500
75%      2420.68500
max      2999.36000
Name: gasto, dtype: float64
2025-01-01 00:00:00 2025-06-29 00:00:00
Duplicados por fecha+pais+canal: 0
Colombia     540
Mexico       540
Argentina    540
Name: pais, dtype: int64
paid_search    540
organic        540
social         540
Name: canal, dtype: int64


In [16]:
print(orders.shape, orders.isna().sum().sum())
print(catalog.shape, catalog.isna().sum().sum())
print(marketing.shape, marketing.isna().sum().sum())

(24950, 12) 410
(7, 4) 0
(1620, 5) 0


## Resumen de la limpieza de datos

Se revisaron los tres datasets (`orders`, `catalog` y `marketing`) en cinco aspectos: formato de fechas, variables numéricas, consistencia de montos, duplicados y variables categóricas.

**Orders.** Las fechas se corrigieron al formato correcto y cubren del 1 de enero al 30 de junio de 2025.
-Se eliminaron 50 filas duplicadas por `id_pedido`, de modo que cada pedido aparece una sola vez. Se encontraron 4 valores negativos en `cantidad`, que se corrigieron como valores absolutos por tratarse de errores de captura. 
-Los valores faltantes se recuperaron cuando fue posible: la categoría del producto se tomó del catálogo (de 80 nulos quedaron 30) y el país y la fuente de referencia se completaron con otros pedidos del mismo usuario (el país pasó de 300 nulos a 19 y la fuente a 0). Los nulos restantes en país, dispositivo, nombre del producto y categoría se marcaron como `desconocido`. 
-50 pedidos (0.2%) tenían `cantidad`, `precio_unitario` y `monto_descuento` vacíos a la vez, por lo que no era posible reconstruirlos a partir de `monto_total` y se eliminaron. Tras estos ajustes, la fórmula `monto_total = cantidad × precio_unitario − monto_descuento` se cumple en todas las filas y no quedan ceros ni negativos en las variables numéricas. 
-Se unificó la escritura del país (2,522 filas venían en minúsculas) y de la categoría `Electrónica`, que aparecía con y sin acento. El dataset final tiene 24,950 filas y ningún valor nulo.

**Catalog.** No presentó duplicados, nulos ni costos inválidos. Solo se unificó la categoría `Electrónica` a `Electronica` para que coincidiera con `orders`. Con este ajuste, la categoría de cada producto es consistente entre ambas tablas y todos los productos vendidos existen en el catálogo.

**Marketing.** Se encontraron 101 valores faltantes en `canal` (6.2%). Como el canal forma parte de `id_campaña`, se recuperó a partir de ese campo, y el resultado fue 540 filas por canal, 540 por país y ningún duplicado por fecha, país y canal. El gasto no tiene valores negativos ni ceros. Los datos cubren del 1 de enero al 29 de junio de 2025, un día menos que `orders`, lo que no afecta el análisis.

**Hallazgo a tener en cuenta.** Al comparar precios con costos, 4,382 pedidos (17.5%) se vendieron por debajo del costo unitario. Todos se concentran en tres productos: Laptop-Gaming-16GB (54% de sus pedidos), Jacket-Winter-M (36%) y Blender-XL-Red (33%). El precio unitario varía entre $20 y $500 en todos los productos sin importar su costo, por lo que estos pedidos no se consideraron errores. Se conservaron en el dataset y se tendrán en cuenta en el análisis de rentabilidad.

In [17]:
print(orders['monto_total'].sum())
print(marketing['gasto'].sum())

51968128.85999999
2871843.53


In [18]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

In [19]:
datasets = {
    'marketing_clean': marketing,   # 'nombre del archivo': variable del notebook
    'orders_clean': orders,
    'catalog_clean': catalog,
}

for nombre, d in datasets.items():
    d.to_excel(f'{nombre}.xlsx', index=False, sheet_name=nombre)
    print(f'Exportado: {nombre}.xlsx ({len(d)} filas, {d.shape[1]} columnas)')

Exportado: marketing_clean.xlsx (1620 filas, 5 columnas)
Exportado: orders_clean.xlsx (24950 filas, 12 columnas)
Exportado: catalog_clean.xlsx (7 filas, 4 columnas)


---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.


In [20]:

# 1. Ingreso total (monto_total ya incluye los descuentos)
revenue = orders['monto_total'].sum()

# 2. Costo total = cantidad vendida x costo unitario del catálogo
df = orders.merge(catalog[['nombre_producto', 'costo_unitario']],
                  on='nombre_producto', how='left')
df['costo_pedido'] = df['cantidad'] * df['costo_unitario']
costo_total = df['costo_pedido'].sum()

# 3. Inversión en marketing
marketing_total = marketing['gasto'].sum()

# 4. Profit y margen
profit = revenue - costo_total - marketing_total
margen = profit / revenue

print(f'Ingreso total:         ${revenue:,.2f}')
print(f'Costo total:           ${costo_total:,.2f}')
print(f'Inversión marketing:   ${marketing_total:,.2f}')
print(f'Profit:                ${profit:,.2f}')
print(f'Margen de ganancia:    {margen:.2%}')
print('¿Es rentable?', 'Sí' if profit > 0 else 'No')

Ingreso total:         $51,968,128.86
Costo total:           $43,124,119.61
Inversión marketing:   $2,871,843.53
Profit:                $5,972,165.72
Margen de ganancia:    11.49%
¿Es rentable? Sí


In [21]:
# Ticket promedio por orden (cada fila es un pedido)
ticket_promedio = orders['monto_total'].mean()

# Cantidad promedio de productos por orden
cantidad_promedio = orders['cantidad'].mean()

# Producto más vendido (por unidades), excluyendo 'desconocido'
ventas_producto = (orders[orders['nombre_producto'] != 'desconocido']
                   .groupby('nombre_producto')['cantidad']
                   .sum()
                   .sort_values(ascending=False))
producto_top = ventas_producto.idxmax()

# Gasto en marketing por canal
gasto_canal = (marketing.groupby('canal')['gasto'].sum() .sort_values(ascending=False))

print(f'Ticket promedio por orden:        ${ticket_promedio:,.2f}')
print(f'Cantidad promedio por orden:      {cantidad_promedio:.2f} productos')
print(f'\nProducto más vendido: {producto_top} ({ventas_producto.max():,.0f} unidades)')
print('\nUnidades vendidas por producto:')
print(ventas_producto)
print('\nGasto en marketing por canal:')
print(gasto_canal.map('${:,.2f}'.format))

Ticket promedio por orden:        $2,082.89
Cantidad promedio por orden:      7.12 productos

Producto más vendido: Laptop-Gaming-16GB (144,198 unidades)

Unidades vendidas por producto:
nombre_producto
Laptop-Gaming-16GB      144198
Vacuum-Pro-Black          6284
Blender-XL-Red            6279
Jacket-Winter-M           6256
Sneakers-Urban-42         6172
Tablet-Standard-64GB      4153
Phone-Pro-128GB           4145
Name: cantidad, dtype: int64

Gasto en marketing por canal:
canal
social         $976,818.37
organic        $972,650.96
paid_search    $922,374.20
Name: gasto, dtype: object


---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.

---

In [22]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [23]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [24]:
events.columns
events['nombre_evento'].unique()

query_funnel = '''
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios
FROM events
GROUP BY nombre_evento
ORDER BY usuarios DESC;
'''

funnel = pd.read_sql(query_funnel, con=engine)
funnel

,nombre_evento,usuarios
0,first_visit,7796
1,add_to_cart,7634
2,select_item,7582
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [25]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios
FROM events
WHERE nombre_evento IN ('first_visit', 'select_item', 'add_to_cart',
                        'begin_checkout', 'add_payment_info', 'purchase')
GROUP BY nombre_evento
ORDER BY CASE nombre_evento
    WHEN 'first_visit'      THEN 1
    WHEN 'select_item'      THEN 2
    WHEN 'add_to_cart'      THEN 3
    WHEN 'begin_checkout'   THEN 4
    WHEN 'add_payment_info' THEN 5
    WHEN 'purchase'         THEN 6
END;
'''
totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,usuarios
0,first_visit,7796
1,select_item,7582
2,add_to_cart,7634
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [26]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH totals AS (
    SELECT
        CASE nombre_evento
            WHEN 'first_visit'      THEN 1
            WHEN 'select_item'      THEN 2
            WHEN 'add_to_cart'      THEN 3
            WHEN 'begin_checkout'   THEN 4
            WHEN 'add_payment_info' THEN 5
            WHEN 'purchase'         THEN 6
        END AS paso,
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios
    FROM events
    WHERE nombre_evento IN ('first_visit', 'select_item', 'add_to_cart',
                            'begin_checkout', 'add_payment_info', 'purchase')
    GROUP BY nombre_evento
)
SELECT
    paso,
    nombre_evento,
    usuarios,
    LAG(usuarios) OVER (ORDER BY paso) - usuarios AS usuarios_perdidos,
    ROUND(100.0 * usuarios / LAG(usuarios) OVER (ORDER BY paso), 2) AS conv_vs_paso_anterior,
    ROUND(100.0 * usuarios / FIRST_VALUE(usuarios) OVER (ORDER BY paso), 2) AS conv_vs_inicio
FROM totals
ORDER BY paso;
'''
conversion = pd.read_sql(query_conversion, con=engine)
conversion

,paso,nombre_evento,usuarios,usuarios_perdidos,conv_vs_paso_anterior,conv_vs_inicio
0,1,first_visit,7796,NaN,NaN,100.00
1,2,select_item,7582,214.0,97.26,97.26
2,3,add_to_cart,7634,-52.0,100.69,97.92
3,4,begin_checkout,7208,426.0,94.42,92.46
4,5,add_payment_info,6250,958.0,86.71,80.17
5,6,purchase,6240,10.0,99.84,80.04


In [27]:
query_conversion = '''
WITH first_visit AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'first_visit'
),
select_item AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'select_item'
),
add_to_cart AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'add_to_cart'
),
begin_checkout AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'begin_checkout'
),
add_payment_info AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'add_payment_info'
),
purchase AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'purchase'
),
funnel_counts AS (
    SELECT
        COUNT(fv.id_usuario)  AS usuarios_first_visit,
        COUNT(si.id_usuario)  AS usuarios_select_item,
        COUNT(a.id_usuario)   AS usuarios_add_to_cart,
        COUNT(bc.id_usuario)  AS usuarios_begin_checkout,
        COUNT(api.id_usuario) AS usuarios_add_payment_info,
        COUNT(p.id_usuario)   AS usuarios_purchase
    FROM first_visit fv
    LEFT JOIN select_item si       ON fv.id_usuario = si.id_usuario
    LEFT JOIN add_to_cart a        ON fv.id_usuario = a.id_usuario
    LEFT JOIN begin_checkout bc    ON fv.id_usuario = bc.id_usuario
    LEFT JOIN add_payment_info api ON fv.id_usuario = api.id_usuario
    LEFT JOIN purchase p           ON fv.id_usuario = p.id_usuario
)
SELECT
    usuarios_first_visit,
    usuarios_select_item,
    usuarios_add_to_cart,
    usuarios_begin_checkout,
    usuarios_add_payment_info,
    usuarios_purchase,
    ROUND(usuarios_select_item     * 100.0 / NULLIF(usuarios_first_visit, 0), 2) AS conversion_select_item,
    ROUND(usuarios_add_to_cart     * 100.0 / NULLIF(usuarios_first_visit, 0), 2) AS conversion_add_to_cart,
    ROUND(usuarios_begin_checkout  * 100.0 / NULLIF(usuarios_first_visit, 0), 2) AS conversion_begin_checkout,
    ROUND(usuarios_add_payment_info * 100.0 / NULLIF(usuarios_first_visit, 0), 2) AS conversion_add_payment_info,
    ROUND(usuarios_purchase        * 100.0 / NULLIF(usuarios_first_visit, 0), 2) AS conversion_purchase
FROM funnel_counts;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,usuarios_first_visit,usuarios_select_item,usuarios_add_to_cart,usuarios_begin_checkout,usuarios_add_payment_info,usuarios_purchase,conversion_select_item,conversion_add_to_cart,conversion_begin_checkout,conversion_add_payment_info,conversion_purchase
0,7796,7393,7439,7031,6088,6068,94.83,95.42,90.19,78.09,77.83


In [28]:
#por país
query_conversion_pais = '''
WITH first_visit AS (
    SELECT DISTINCT id_usuario, pais
    FROM events
    WHERE nombre_evento = 'first_visit'
),
select_item AS (
    SELECT DISTINCT id_usuario, pais
    FROM events
    WHERE nombre_evento = 'select_item'
),
add_to_cart AS (
    SELECT DISTINCT id_usuario, pais
    FROM events
    WHERE nombre_evento = 'add_to_cart'
),
begin_checkout AS (
    SELECT DISTINCT id_usuario, pais
    FROM events
    WHERE nombre_evento = 'begin_checkout'
),
add_payment_info AS (
    SELECT DISTINCT id_usuario, pais
    FROM events
    WHERE nombre_evento = 'add_payment_info'
),
purchase AS (
    SELECT DISTINCT id_usuario, pais
    FROM events
    WHERE nombre_evento = 'purchase'
),
funnel_counts AS (
    SELECT
        fv.pais,
        COUNT(fv.id_usuario)  AS usuarios_first_visit,
        COUNT(si.id_usuario)  AS usuarios_select_item,
        COUNT(a.id_usuario)   AS usuarios_add_to_cart,
        COUNT(bc.id_usuario)  AS usuarios_begin_checkout,
        COUNT(api.id_usuario) AS usuarios_add_payment_info,
        COUNT(p.id_usuario)   AS usuarios_purchase
    FROM first_visit fv
    LEFT JOIN select_item si       ON fv.id_usuario = si.id_usuario  AND fv.pais = si.pais
    LEFT JOIN add_to_cart a        ON fv.id_usuario = a.id_usuario   AND fv.pais = a.pais
    LEFT JOIN begin_checkout bc    ON fv.id_usuario = bc.id_usuario  AND fv.pais = bc.pais
    LEFT JOIN add_payment_info api ON fv.id_usuario = api.id_usuario AND fv.pais = api.pais
    LEFT JOIN purchase p           ON fv.id_usuario = p.id_usuario   AND fv.pais = p.pais
    GROUP BY fv.pais
)
SELECT
    pais,
    usuarios_first_visit,
    usuarios_purchase,
    ROUND(usuarios_select_item      * 100.0 / NULLIF(usuarios_first_visit, 0), 2) AS conversion_select_item,
    ROUND(usuarios_add_to_cart      * 100.0 / NULLIF(usuarios_first_visit, 0), 2) AS conversion_add_to_cart,
    ROUND(usuarios_begin_checkout   * 100.0 / NULLIF(usuarios_first_visit, 0), 2) AS conversion_begin_checkout,
    ROUND(usuarios_add_payment_info * 100.0 / NULLIF(usuarios_first_visit, 0), 2) AS conversion_add_payment_info,
    ROUND(usuarios_purchase         * 100.0 / NULLIF(usuarios_first_visit, 0), 2) AS conversion_purchase
FROM funnel_counts
ORDER BY conversion_purchase DESC;
'''

conversion_pais = pd.read_sql(query_conversion_pais, con=engine)
conversion_pais

,pais,usuarios_first_visit,usuarios_purchase,conversion_select_item,conversion_add_to_cart,conversion_begin_checkout,conversion_add_payment_info,conversion_purchase
0,Argentina,2543,2012,95.08,95.79,90.05,77.51,79.12
1,Colombia,2635,2052,94.46,95.26,90.74,78.33,77.87
2,Mexico,2618,2004,94.96,95.23,89.76,78.42,76.55


## Conclusiones

### Parte 1: Construcción del funnel

Se construyó el funnel como una cohorte: se tomaron los usuarios únicos con `first_visit` y se contó cuántos de ellos llegaron a cada etapa posterior (LEFT JOIN por `id_usuario`).

| Etapa | Usuarios únicos |
|---|---|
| first_visit | 7,796 |
| select_item | 7,393 |
| add_to_cart | 7,439 |
| begin_checkout | 7,031 |
| add_payment_info | 6,088 |
| purchase | 6,068 |

**Nota:** `add_to_cart` tiene más usuarios que `select_item` (7,439 vs. 7,393), por lo que algunos usuarios agregan al carrito sin registrar `select_item`. Además, entre 162 y 195 usuarios por etapa registran eventos posteriores sin tener `first_visit`, y quedan fuera de esta cohorte.

### Parte 2: Análisis de conversión

| Paso | Conversión vs. paso anterior |
|---|---|
| first_visit → select_item | 94.83% |
| select_item → add_to_cart | 100.62% (anomalía) |
| add_to_cart → begin_checkout | 94.51% |
| begin_checkout → add_payment_info | 86.59% |
| add_payment_info → purchase | 99.67% |

**¿En qué etapa se pierde la mayor cantidad de usuarios?**
Entre `begin_checkout` y `add_payment_info`: se pierden 943 usuarios (13.41%), la caída más fuerte del funnel. Una hipótesis es que haya fricción al ingresar la información de pago, aunque los datos no permiten confirmar la causa.

**¿Cuál es la tasa de conversión final?**
**77.84%**: de 7,796 usuarios que iniciaron, 6,068 completaron la compra. Una vez que el usuario agrega su información de pago, casi siempre compra (99.67%).

---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.


In [29]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [30]:
#conversion de fecha "users"
query_users = '''
SELECT
    id_usuario,
    CAST(fecha_registro AS DATE) AS fecha_registro,
    "país",
    dispositivo,
    tipo_plan
FROM public.users;
'''
users = pd.read_sql(query_users, con=engine)

In [31]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [32]:
#conversion de fecha "user_activity"
query_activity = '''
SELECT
    id_usuario,
    CAST(fecha_actividad AS DATE) AS fecha_actividad,
    dias_despues_registro,
    activo
FROM public.user_activity;
'''
user_activity = pd.read_sql(query_activity, con=engine)

In [33]:
print('Registro:  ', users['fecha_registro'].min(), '->', users['fecha_registro'].max())
print('Actividad: ', user_activity['fecha_actividad'].min(), '->', user_activity['fecha_actividad'].max())

Registro:   2025-01-01 -> 2025-05-31
Actividad:  2025-01-08 -> 2025-06-28


In [34]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH cohortes AS (
    SELECT
        id_usuario,
        CAST(DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS DATE) AS cohorte
    FROM public.users
),
actividad AS (
    SELECT DISTINCT
        id_usuario,
        CASE
            WHEN dias_despues_registro BETWEEN 1  AND 7  THEN 1
            WHEN dias_despues_registro BETWEEN 8  AND 14 THEN 2
            WHEN dias_despues_registro BETWEEN 15 AND 21 THEN 3
        END AS semana
    FROM public.user_activity
    WHERE activo = 1
      AND dias_despues_registro BETWEEN 1 AND 21
)
SELECT
    c.cohorte,
    COUNT(DISTINCT c.id_usuario) AS usuarios_cohorte,
    COUNT(DISTINCT CASE WHEN a.semana = 1 THEN a.id_usuario END) AS retenido_w1,
    COUNT(DISTINCT CASE WHEN a.semana = 2 THEN a.id_usuario END) AS retenido_w2,
    COUNT(DISTINCT CASE WHEN a.semana = 3 THEN a.id_usuario END) AS retenido_w3
FROM cohortes c
LEFT JOIN actividad a ON c.id_usuario = a.id_usuario
GROUP BY c.cohorte
ORDER BY c.cohorte;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte,usuarios_cohorte,retenido_w1,retenido_w2,retenido_w3
0,2025-01-01,1627,697,668,656
1,2025-02-01,1444,611,609,635
2,2025-03-01,1636,677,705,690
3,2025-04-01,1606,680,697,663
4,2025-05-01,1687,695,676,706


In [35]:
#Porcentaje de retención
for n in [1, 2, 3]:
    cohorte_final[f'semana_{n}'] = (cohorte_final[f'retenido_w{n}'] / cohorte_final['usuarios_cohorte'] * 100).round(2)

cohorte_final

,cohorte,usuarios_cohorte,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01-01,1627,697,668,656,42.84,41.06,40.32
1,2025-02-01,1444,611,609,635,42.31,42.17,43.98
2,2025-03-01,1636,677,705,690,41.38,43.09,42.18
3,2025-04-01,1606,680,697,663,42.34,43.40,41.28
4,2025-05-01,1687,695,676,706,41.20,40.07,41.85


---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.


---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** la tasa de conversión del grupo variante es igual a la del grupo control (p_variante = p_control).
   - **H₁ (Hipótesis alternativa):** la tasa de conversión es diferente entre ambos grupos (p_variante ≠ p_control).
     
**Test estadístico:** Z-test

**Nivel de significancia alpha:** α = 0.05 (test de dos colas).


In [36]:
import pandas as pd

df = pd.read_csv('/datasets/experiment_checkout_ui.csv')

print(df.shape)
print(df.columns.tolist())
df.head()

(10000, 7)
['id_usuario', 'variante', 'convirtio', 'dispositivo', 'pais', 'duracion_sesion', 'timestamp']


,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [37]:

resumen = df.groupby('variante')['convirtio'].agg(
    usuarios='count',
    compras='sum',
    tasa_conversion='mean'
)
resumen['tasa_conversion'] = (resumen['tasa_conversion'] * 100).round(2)
resumen

,usuarios,compras,tasa_conversion
variante,,,
control,4965,779,15.69
tratamiento,5035,820,16.29


In [38]:
print(df['id_usuario'].duplicated().sum())   # esperado: 0
print(df['convirtio'].value_counts())        # solo 0 y 1

0
0    8401
1    1599
Name: convirtio, dtype: int64


In [39]:
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

alpha = 0.05

compras = [resumen.loc['tratamiento', 'compras'], resumen.loc['control', 'compras']]
usuarios = [resumen.loc['tratamiento', 'usuarios'], resumen.loc['control', 'usuarios']]

z, p_value = proportions_ztest(count=compras, nobs=usuarios, alternative='two-sided')

p_trat = compras[0] / usuarios[0]
p_ctl = compras[1] / usuarios[1]
low, high = confint_proportions_2indep(compras[0], usuarios[0], compras[1], usuarios[1], method='wald')

print(f'Control:     {p_ctl:.2%}')
print(f'Tratamiento: {p_trat:.2%}')
print(f'Diferencia:  {(p_trat - p_ctl) * 100:.2f} puntos porcentuales')
print(f'IC 95% de la diferencia: [{low * 100:.2f}, {high * 100:.2f}] pp')
print(f'z = {z:.3f}, p-value = {p_value:.4f}')

if p_value < alpha:
    print('Se rechaza H0: la diferencia es estadísticamente significativa.')
else:
    print('No se rechaza H0: no hay evidencia suficiente de diferencia.')

Control:     15.69%
Tratamiento: 16.29%
Diferencia:  0.60 puntos porcentuales
IC 95% de la diferencia: [-0.84, 2.03] pp
z = 0.813, p-value = 0.4161
No se rechaza H0: no hay evidencia suficiente de diferencia.


## Conclusiones: test estadístico

Se evaluó con un z-test de dos proporciones (dos colas, α = 0.05).

| Grupo | Usuarios | Compras | Conversión |
|---|---|---|---|
| Control | 4,965 | 779 | 15.69% |
| Tratamiento | 5,035 | 820 | 16.29% |

- El p-value es 0.4161, mayor que 0.05, por lo que **no se rechaza H0**: no hay evidencia estadísticamente significativa de que la nueva UI modifique la conversión.

**Recomendación:** con estos datos no se justifica atribuir una mejora a la nueva UI. Si el equipo considera relevante un efecto pequeño, se necesitaría un experimento más grande o más largo para detectarlo.

---

## 🔹 Paso 6: Comunicar los resultados (Dashboard)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 


---

# Dashboard:
https://public.tableau.com:9443/views/Proyectofinal_17951660793520/OverviewEjecutivo?:language=es-ES&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link
